In [45]:
import pandas as pd
path = "../data/raw/cmu_books/booksummaries.txt"

df = pd.read_csv(path, sep="\t", header=None)
print(df.head())
print(df.shape)
print(df.info())
print(df.isnull().sum())

      0        1                                          2                3  \
0   620  /m/0hhy                                Animal Farm    George Orwell   
1   843  /m/0k36                         A Clockwork Orange  Anthony Burgess   
2   986  /m/0ldx                                 The Plague     Albert Camus   
3  1756  /m/0sww  An Enquiry Concerning Human Understanding       David Hume   
4  2080  /m/0wkt                       A Fire Upon the Deep     Vernor Vinge   

            4                                                  5  \
0  1945-08-17  {"/m/016lj8": "Roman \u00e0 clef", "/m/06nbt":...   
1        1962  {"/m/06n90": "Science Fiction", "/m/0l67h": "N...   
2        1947  {"/m/02m4t": "Existentialism", "/m/02xlf": "Fi...   
3         NaN                                                NaN   
4         NaN  {"/m/03lrw": "Hard science fiction", "/m/06n90...   

                                                   6  
0   Old Major, the old boar on the Manor Farm, ca...  


In [46]:
import sys
from pathlib import Path

import pandas as pd

ROOT = Path.cwd()

# Works if the notebook is opened from the project root
if not (ROOT / "data").exists():
    ROOT = ROOT.parent

sys.path.insert(0, str(ROOT / "backend"))

books_path = ROOT / "data" / "processed" / "books.parquet"

books = pd.read_parquet(books_path)

print("Dataset shape:", books.shape)
print("Columns:", books.columns.tolist())
print("Empty Genres: ", books["genres"].map(len).eq(0).sum())
print("Empty Authors: ", (books["author"] == "").sum())
print("Missing Dates: ", (books["publication_date"] == "").sum())


Dataset shape: (16551, 8)
Columns: ['wiki_id', 'freebase_id', 'title', 'author', 'publication_date', 'genres', 'summary', 'embedding_text']
Empty Genres:  3717
Empty Authors:  2379
Missing Dates:  5606


In [47]:
# reading 20 rows by hand
print(books.sample(20))


        wiki_id freebase_id  \
14907  25973624  /m/0b6lm8n   
16176  32112497  /m/0gx09bw   
7478    6745182   /m/0glsqz   
8655    8700946  /m/027fgpb   
10115  11850869  /m/02rvlnh   
3650    2449341   /m/07dx1b   
6952    5856785   /m/0f99tq   
15003  26346398  /m/06jmzvg   
5947    4904241   /m/0ctf44   
15264  27441591  /m/0c02xsd   
10857  13656504  /m/04ggf0y   
5732    4629105   /m/0cdgsg   
8371    8188949  /m/026vmyt   
15002  26342598   /m/04hwsb   
15400  28138267  /m/0cm8y2b   
6139    5069054   /m/0d19b6   
12032  16958903  /m/0413dlg   
3219    2061112   /m/06jbyc   
5128    3979496   /m/0b9t77   
16445  34052275  /m/0hr1rjc   

                                                   title              author  \
14907  Praying for Power: Buddhism and the Formation ...       Timothy Brook   
16176                                     The Blood Book          Ted Dekker   
7478                     Agent Arthur's Desert Challenge       Martin Oliver   
8655                        

In [48]:
print("Embedding text: ", books["embedding_text"].iloc[0])

Embedding text:  Animal Farm by George Orwell. Genres: Children's literature, Fiction, Roman à clef, Satire, Speculative fiction. Old Major, the old boar on the Manor Farm, calls the animals on the farm for a meeting, where he compares the humans to parasites and teaches the animals a revolutionary song, 'Beasts of England'. When Major dies, two young pigs, Snowball and Napoleon, assume command and turn his dream into a philosophy. The animals revolt and drive the drunken and irresponsible Mr Jones from the farm, renaming it "Animal Farm". They adopt Seven Commandments of Animal-ism, the most important of which is, "All animals are equal". Snowball attempts to teach the animals reading and writing; food is plentiful, and the farm runs smoothly. The pigs elevate themselves to positions of leadership and set aside special food items, ostensibly for their personal health. Napoleon takes the pups from the farm dogs and trains them privately. Napoleon and Snowball struggle for leadership. W

In [49]:
from app.core.embeddings import embed_documents
from app.store import BruteForceStore

vectors = embed_documents(books["embedding_text"].tolist())

documents = books[["wiki_id", "title", "author", "genres"]].to_dict("records")
for document in documents:
    document["id"] = str(document.pop("wiki_id"))

store = BruteForceStore()
store.add_documents(documents, vectors)

print(vectors.shape, len(store))

Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

Batches:   0%|          | 0/518 [00:00<?, ?it/s]

(16551, 384) 16551


In [50]:
def show(results):
    rows = []
    for result in results:
        row = dict(result["document"])
        row["score"] = result["score"]
        rows.append(row)
    return pd.DataFrame(rows)


def similar_to_book(title, k=5):
    row = books[books["title"] == title].iloc[0]
    book_id = str(row["wiki_id"])
    results = store.search(store.get_vector(book_id), k=k, exclude_id=book_id)
    return show(results)

In [51]:
similar_to_book("Animal Farm")

,title,author,genres,id,score
0,Snowball's Chance,John Reed,[Parody],11541759,0.750931
1,The Sheep-Pig,Dick King-Smith,"[Children's literature, Fiction]",3021563,0.624859
2,Punk Farm,Jarrett Krosoczka,[Children's literature],4656302,0.565044
3,George's Marvelous Medicine,Roald Dahl,"[Children's literature, Speculative fiction]",1117368,0.543360
4,The Tale of Pigling Bland,Beatrix Potter,"[Children's literature, Fiction]",10309074,0.526906


In [52]:
from app.core.embeddings import embed_query


def search(query, k=10):
    return show(store.search(embed_query(query), k=k))


search("a cozy mystery set in a small village")

,title,author,genres,id,score
0,The Jungle Pyramid,Franklin W. Dixon,"[Detective fiction, Mystery]",2074207,0.461621
1,The Castle of Adventure,Enid Blyton,[Adventure novel],23943721,0.445913
2,Irish Stew!,Andrew Greeley,"[Mystery, Speculative fiction]",12089587,0.444605
3,The Mystery of the Cape Cod Tavern,Phoebe Atwood Taylor,"[Detective fiction, Mystery]",14857283,0.439666
4,The Mysterious Mannequin,Carolyn Keene,"[Children's literature, Detective fiction, Mys...",6900207,0.437488
5,The Kachina Doll Mystery,Carolyn Keene,"[Children's literature, Detective fiction, Fic...",9553753,0.435050
6,Smaller and Smaller Circles,,[],21844678,0.433483
7,The Quest of the Missing Map,Carolyn Keene,"[Children's literature, Detective fiction, Fic...",6899604,0.432799
8,The Secret of the Forgotten City,Carolyn Keene,"[Children's literature, Detective fiction, Mys...",6914967,0.428754
9,Where The Wild Things Are,Maurice Sendak,"[Children's literature, Fantasy, Fiction, Pict...",1229351,0.425336
